<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/7_rewrite_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# vocabulary mismatch problem

# we first use an LLM to transform it into a much richer and more effective one
# HyDE (Hypothetical Document Embeddings)

In [ ]:
import sys
project_root_dir = "/Users/karimelbana/wbs/rag_project_DSAI/"
sys.path.append(project_root_dir)

# Part 7: Evaluating Query Rewriting

This guide provides instructions for the final optimisation stage of our project: evaluating query rewriting. This is an advanced technique that can help the retriever find more relevant information by transforming the user's initial question into a more effective one.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

---
## 1.&nbsp; What is Query Rewriting?

Sometimes, a user's question doesn't contain the best keywords for finding the right information. For example, a user might ask, "What did the sleepy mouse say at the tea party?" The word "mouse" might not appear in the most relevant text chunk, which instead refers to the character as the "Dormouse".

Query rewriting tackles this problem. We'll use a specific technique called HyDE (Hypothetical Document Embeddings). Here's how it works:

1.  **Original Question**: The user asks a question (e.g., "What did the Dormouse say?").
2.  **Generate Hypothetical Answer**: Before searching, the LLM generates a hypothetical, plausible answer to the question. It might generate something like, "The Dormouse, a sleepy character at the Mad Hatter's tea party, told a story about three little sisters who lived at the bottom of a treacle-well."
3.  **Search with the Answer**: The system then converts this longer, more detailed hypothetical answer into a vector embedding and uses that to search the knowledge base.
4.  **Retrieve Real Documents**: Because the hypothetical answer is rich with context and relevant keywords, its embedding is often much better at finding the truly relevant document chunks.

This process helps bridge the vocabulary gap between the user's question and the source documents, often leading to significant improvements in retrieval quality.

---
## 2.&nbsp; Preparing for the Experiment

To test the effect of HyDE, we'll add our final evaluation stage.

### 2.1 Add Query Rewriting Configurations

First, we need to define the baseline for our experiment. This will be the best-performing configuration we have found so far, combining our optimal chunking and reranking strategies.

1.  Open `evaluation/eval_config.py` in your VSCode editor.
2.  Add the following code to the end of the file.

In [ ]:
import pandas as pd
import numpy as np

results = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/reranker_evaluation_summary_2025-09-22_21-47-10.csv')
results

,chunk_size,chunk_overlap,retriever_k,reranker_n,faithfulness,context_precision,context_recall
0,512,50,10,2,0.774510,0.750000,0.750000
1,512,50,10,5,0.819444,0.741667,0.916667
2,512,50,20,5,0.675926,0.741667,0.750000


In [ ]:
results['mean_score'] = results.apply(lambda row: np.mean([
    row['faithfulness'], row['context_precision'], row['context_recall']]
    ),axis = 1)
results

,chunk_size,chunk_overlap,retriever_k,reranker_n,faithfulness,context_precision,context_recall,mean_score
0,512,50,10,2,0.774510,0.750000,0.750000,0.758170
1,512,50,10,5,0.819444,0.741667,0.916667,0.825926
2,512,50,20,5,0.675926,0.741667,0.750000,0.722531


In [ ]:
best_retriever_k = int(results.sort_values(by='mean_score', ascending=False).head(1)['retriever_k'].values[0])
best_retriever_k

10

In [ ]:
best_reranker_n = int(results.sort_values(by='mean_score', ascending=False).head(1)['reranker_n'].values[0])
best_reranker_n

5

In [ ]:
# --- Query Rewrite Evaluation ---
# The 'best' reranker strategy found in the previous evaluation stage.
# IMPORTANT: You must update this with the values you found to be optimal.
# BEST_RERANKER_STRATEGY: dict[str, int] = {'retriever_k': best_retriever_k, 'reranker_n': best_reranker_n}
BEST_RERANKER_STRATEGY: dict[str, int] = {'retriever_k': 10, 'reranker_n': 5}

### 2.2 Create the Query Rewriting Evaluation Function

Now, let's update `evaluation/run_evaluation.py` with the logic to test HyDE.

1.  **Add New Imports**: Open `evaluation/run_evaluation.py` and add the new imports. The `TransformQueryEngine` and `HyDEQueryTransform` are the new components for this stage.

In [ ]:
# Add to existing llama-index.core imports
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core.indices.query.query_transform import HyDEQueryTransform

# Add the new config to the import from evaluation.evaluation_config
from evaluation.eval_config import (
    # ... existing imports
    RERANKER_CONFIGS,
    BEST_RERANKER_STRATEGY,
)

2.  **Add the Evaluation Function**: Add the following function to the file, placing it after the `evaluate_reranker_strategies` function.

In [ ]:
def evaluate_query_rewriting(
    llm: Groq,
    embed_model: HuggingFaceEmbedding
) -> pd.DataFrame:
    """ Evaluates the impact of HyDE on top of the best RAG configuration. """
    print("\n--- 🚀 Stage 4: Evaluating Query Rewriting (HyDE) ---")
    questions, ground_truths = get_eval_data() #1️⃣
    all_results: list[pd.DataFrame] = []

    # Use the best configurations from the config file
    best_chunk_size: int = BEST_CHUNKING_STRATEGY['size'] #512
    best_chunk_overlap: int = BEST_CHUNKING_STRATEGY['overlap'] #50
    best_retriever_k: int = BEST_RERANKER_STRATEGY['retriever_k'] #10
    best_reranker_n: int = BEST_RERANKER_STRATEGY['reranker_n'] #5

    index: VectorStoreIndex = get_or_build_index( #2️⃣
        chunk_size=best_chunk_size,
        chunk_overlap=best_chunk_overlap,
        embed_model=embed_model
    )

    # Test with and without HyDE
    for use_hyde in [False, True]:
        print(f"\n--- Testing Query Rewrite Config: use_hyde={use_hyde} ---")

        # Build the base query engine with retriever and reranker #3️⃣
        retriever: VectorIndexRetriever = index.as_retriever( #3️⃣.1️⃣
            similarity_top_k=best_retriever_k
        )
        reranker: SentenceTransformerRerank = SentenceTransformerRerank( #3️⃣.2️⃣
            top_n=best_reranker_n, model=RERANKER_MODEL_NAME
        )
        base_query_engine: RetrieverQueryEngine = RetrieverQueryEngine.from_args( #3️⃣.3️⃣
            retriever=retriever, node_postprocessors=[reranker], llm=llm
        )

        if use_hyde: #3️⃣
            hyde_transform = HyDEQueryTransform(include_original=True) #3️⃣.1️⃣

            # TransformQueryEngine: when the query_engine receives the query,
            # before doing cosine_similarity: HyDE transform the query
            query_engine = TransformQueryEngine(base_query_engine, query_transform=hyde_transform)#3️⃣.2️⃣

            # query_engine = TransformQueryEngine(base_query_engine, query_transform=HyDEQueryTransform(include_original=True))#3️⃣.2️⃣

        else:
            # When not using HyDE, the engine is just the base engine
            query_engine = base_query_engine

        qa_dataset: Dataset = generate_qa_dataset(query_engine, questions, ground_truths)#4️⃣

        print("--- Running Ragas evaluation for query rewriting... ---")
        result = evaluate(dataset=qa_dataset, metrics=EVAL_METRICS, raise_exceptions=True)#5️⃣

        results_df: pd.DataFrame = result.to_pandas()
        results_df['chunk_size'] = best_chunk_size
        results_df['chunk_overlap'] = best_chunk_overlap
        results_df['retriever_k'] = best_retriever_k
        results_df['reranker_n'] = best_reranker_n
        results_df['use_hyde'] = use_hyde
        all_results.append(results_df)

    final_df: pd.DataFrame = pd.concat(all_results, ignore_index=True)
    save_results(final_df, "query_rewrite_evaluation")
    print("--- ✅ Query Rewrite Evaluation Complete ---")
    return final_df

This function first builds our best-performing RAG engine (with a retriever and reranker) and then runs the evaluation twice: once with the base engine, and once with the engine wrapped in a `TransformQueryEngine` that applies the HyDE transformation.

3.  **Update the `save_results` Helper**: To get a correct summary file, we must tell our `save_results` function about the new `use_hyde` parameter.

In [ ]:
# In the save_results function:
param_cols: list[str] = [
    col
    for col in ['chunk_size', 'chunk_overlap', 'retriever_k', 'reranker_n', 'use_hyde']
    if col in results_df.columns
]

### 2.3 Update the Main Execution Block

Finally, let's update the main execution block to run our new stage. Remember to comment out the previous stages.

In `evaluation/run_evaluation.py`, update the `if __name__ == "__main__":` block:

In [ ]:
if __name__ == "__main__":
    llm_to_test: Groq = initialise_llm()
    embed_model_to_test: HuggingFaceEmbedding = get_embedding_model()

    # Comment out previous stages
    # evaluate_baseline(llm=llm_to_test, embed_model=embed_model_to_test)
    # evaluate_chunking_strategies(llm=llm_to_test, embed_model=embed_model_to_test)
    # evaluate_reranker_strategies(llm=llm_to_test, embed_model=embed_model_to_test)

    # Run Stage 4: Query Rewrite Evaluation
    evaluate_query_rewriting(llm=llm_to_test, embed_model=embed_model_to_test)

---
## 3.&nbsp; Run the Query Rewriting Evaluation

With the new function and configurations in place, you are ready to run the experiment.

From your VSCode terminal, run the evaluation script as before:

In [ ]:
python -m evaluation.run_evaluation

The script will now run the evaluation twice: once with your best RAG pipeline, and once with HyDE enabled on top of it.

---
## 4.&nbsp; Analyse the Results

Once the script finishes, navigate to the `evaluation/evaluation_results` folder. You will find the new `query_rewrite_evaluation_summary_... .csv` file.

Open this summary file. It will contain two rows. Compare the scores for `use_hyde: False` (your best pipeline so far) with `use_hyde: True`. Does adding HyDE improve the scores? Pay close attention to **`context_recall`**, as this is the metric HyDE is most designed to improve.

---
## 5.&nbsp; Challenges and Further Exploration 😀

You've now evaluated a full suite of advanced RAG techniques!

### 1. Find Your Optimal Query Rewriting Strategy

It's time to run the query rewriting evaluation on your custom RAG system and determine if HyDE is beneficial for your specific documents and questions.

**Your Mission:**

1.  **Confirm Your Baseline**: Double-check that `BEST_CHUNKING_STRATEGY` and `BEST_RERANKER_STRATEGY` in `evaluation_config.py` are set to the optimal values you found for your project.
2.  **Run the Query Rewriting Evaluation**: Execute `python -m evaluation.run_evaluation`.
3.  **Set Up Your Analysis Notebook**: In your `notebooks` directory, create a new Notebook file named `04_query_rewrite_analysis.ipynb`.
4.  **Load and Analyse Your Results**:
    * In the new notebook, load the summary results from your query rewriting experiment.
    * Analyse the results. Does HyDE provide a significant improvement for your use case? Document your conclusions in the notebook.

In [ ]:
import seaborn as sns

In [ ]:
titanic = sns.load_dataset('titanic')
titanic.head(1)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.25,S,Third,man,True,NaN,Southampton,no,False


In [ ]:
gender = titanic[['sex', 'who']]
gender

,sex,who
0,male,man
1,female,woman
2,female,woman
3,female,woman
4,male,man
...,...,...
886,male,man
887,female,woman
888,female,woman
889,male,man


In [ ]:
gender[gender['sex'] == 'male']['who'].unique()

array(['man', 'child'], dtype=object)

In [ ]:
titanic.groupby('class').mean(numeric_only=True)

/var/folders/b7/8b0bdwr52qj1ys_pgmnv_8mc0000gn/T/ipykernel_67828/700458660.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby('class').mean(numeric_only=True)


,survived,pclass,age,sibsp,parch,fare,adult_male,alone
class,,,,,,,,
First,0.629630,1.0,38.233441,0.416667,0.356481,84.154687,0.550926,0.504630
Second,0.472826,2.0,29.877630,0.402174,0.380435,20.662183,0.538043,0.565217
Third,0.242363,3.0,25.140620,0.615071,0.393075,13.675550,0.649695,0.659878


In [ ]:
import pandas as pd
import numpy as np

results_rewritng = pd.read_csv('/Users/karimelbana/wbs/rag_project_DSAI/evaluation/evaluation_results/query_rewrite_evaluation_summary_2025-09-22_23-00-59.csv')
results_rewritng

,chunk_size,chunk_overlap,retriever_k,reranker_n,use_hyde,faithfulness,context_precision,context_recall
0,512,50,10,5,False,0.736111,0.741667,0.833333
1,512,50,10,5,True,0.934524,0.750000,0.916667


In [ ]:
results_rewritng['mean_score'] = results_rewritng.apply(lambda row: np.mean([
    row['faithfulness'], row['context_precision'], row['context_recall']]
    ),axis = 1)
results_rewritng

,chunk_size,chunk_overlap,retriever_k,reranker_n,use_hyde,faithfulness,context_precision,context_recall,mean_score
0,512,50,10,5,False,0.736111,0.741667,0.833333,0.770370
1,512,50,10,5,True,0.934524,0.750000,0.916667,0.867063
